# 18 — Top-K Heavy Hitters (Amazon FAR style)

Find the most frequent items in a stream ("trending", top talkers, hot keys). Parts 1-3 build the core
(concurrency at Part 3); Parts 4-5 are stretch tasks (approximate heavy hitters in bounded memory, then
a parallel map-reduce). Fill stubs, run each test cell, peek at the collapsed `ref_` solutions only
after trying.

---

## Part 1 — Count frequencies

`count(items) -> dict[item, freq]`.

**Lock down:** the trivial base; everything else is "what do you do when the counts don't fit in
memory / arrive concurrently / you only want the top few."

In [ ]:
def count(items):
    raise NotImplementedError

In [ ]:
# --- Part 1 tests ---
def _t1():
    assert count(["a", "b", "a", "c", "a", "b"]) == {"a": 3, "b": 2, "c": 1}
    assert count([]) == {}
    print("Part 1 OK")

_t1()

## Part 2 — Top-K

`top_k(counts, k) -> list[(item, freq)]`: the `k` highest-frequency items, sorted by frequency
descending, ties broken by item ascending (deterministic).

**Lock down:** a heap gives O(n log k) without sorting everything; deterministic tie-breaking matters
for testability.

In [ ]:
def top_k(counts, k):
    raise NotImplementedError

In [ ]:
# --- Part 2 tests ---
def _t2():
    counts = {"a": 5, "b": 3, "c": 5, "d": 1}
    assert top_k(counts, 2) == [("a", 5), ("c", 5)]    # tie a<c
    assert top_k(counts, 3) == [("a", 5), ("c", 5), ("b", 3)]
    assert top_k({}, 3) == []
    print("Part 2 OK")

_t2()

## Part 3 — Concurrent counter

`ConcurrentCounter`: thread-safe `add(item, n=1)`, `top_k(k)`, and `total(item)`. Many threads update
overlapping items at once with no lost increments.

**The invariant:** 8 threads each `add("x")` and `add("y")` 1000 times leaves `total("x") ==
total("y") == 8000`. **Discuss:** the read-modify-write race on each counter; striped/sharded locks to
reduce contention; snapshotting for `top_k`.

In [ ]:
import threading


class ConcurrentCounter:
    def __init__(self):
        raise NotImplementedError

    def add(self, item, n=1):
        raise NotImplementedError

    def total(self, item):
        raise NotImplementedError

    def top_k(self, k):
        raise NotImplementedError

In [ ]:
# --- Part 3 tests ---
def _t3():
    cc = ConcurrentCounter()

    def worker():
        for _ in range(1000):
            cc.add("x"); cc.add("y")

    ts = [threading.Thread(target=worker) for _ in range(8)]
    for t in ts: t.start()
    for t in ts: t.join()
    assert cc.total("x") == 8000 and cc.total("y") == 8000
    assert cc.top_k(1) in ([("x", 8000)], [("y", 8000)])   # tie either way, both 8000
    print("Part 3 OK")

_t3()

## Part 4 (stretch) — Misra–Gries approximate heavy hitters

When the key space is huge you can't keep an exact count. `heavy_hitters(stream, k) -> dict` runs the
**Misra–Gries** algorithm with at most `k-1` counters: increment if tracked; else start a counter if
there's room; else decrement *all* counters (dropping any that hit 0). It guarantees the result is a
**superset of every item with frequency > n/k** (with bounded over-/under-counts).

**Discuss:** the guarantee (no true heavy hitter is missed; some false positives possible), bounded
memory `k-1`, and the streaming/approximation trade-off — the recurring theme.

In [ ]:
def heavy_hitters(stream, k):
    raise NotImplementedError

In [ ]:
# --- Part 4 tests ---
def _t4():
    stream = ["a"] * 60 + ["b"] * 10 + ["c"] * 10 + ["d"] * 10 + ["e"] * 10   # n=100
    hh = heavy_hitters(stream, k=4)                      # threshold n/k = 25
    assert len(hh) <= 3                                  # at most k-1 counters
    n = len(stream)
    true_hh = {x for x in set(stream) if stream.count(x) > n // 4}
    assert true_hh <= set(hh)                            # guarantee: superset of true heavy hitters
    assert "a" in hh                                     # a (60 > 25) must be present
    print("Part 4 OK")

_t4()

## Part 5 (stretch) — Parallel map-reduce top-K

`parallel_top_k(items, k, num_procs) -> list[(item, freq)]`: count a large item list across processes
with `ProcessPoolExecutor` (worker `heavyhitters_workers.count_chunk`), merge the partial counts, and
return the global top-`k` (same ordering as Part 2).

**Discuss:** map (per-chunk counts) then reduce (merge) then top-k; GIL (counting is CPU-bound ->
processes); and that exact top-k needs the full merged counts, while Part 4 trades exactness for memory.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import heavyhitters_workers


def parallel_top_k(items, k, num_procs):
    raise NotImplementedError

In [ ]:
# --- Part 5 tests ---
def _t5():
    import random
    items = ["a"] * 30 + ["b"] * 20 + ["c"] * 10
    random.Random(0).shuffle(items)
    assert parallel_top_k(items, 2, num_procs=4) == [("a", 30), ("b", 20)]
    print("Part 5 OK")

_t5()

## Likely follow-ups
- Count–Min Sketch (sublinear frequency estimate) + a heap of candidates; Space-Saving algorithm.
- Sliding-window / time-decayed top-k (trending now vs all-time).
- Distributed top-k: per-shard top-k is *not* exact globally — why, and how to fix (send extra).

---
## Reference solutions (don't peek until you've tried)

In [ ]:
import threading
from concurrent.futures import ProcessPoolExecutor
import heavyhitters_workers


def ref_count(items):
    c = {}
    for x in items:
        c[x] = c.get(x, 0) + 1
    return c


def ref_top_k(counts, k):
    return sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))[:k]


class RefConcurrentCounter:
    def __init__(self):
        self.c = {}
        self.lock = threading.Lock()

    def add(self, item, n=1):
        with self.lock:
            self.c[item] = self.c.get(item, 0) + n
    def total(self, item):
        with self.lock:
            return self.c.get(item, 0)

    def top_k(self, k):
        with self.lock:
            return sorted(self.c.items(), key=lambda kv: (-kv[1], kv[0]))[:k]


def ref_heavy_hitters(stream, k):
    counters = {}
    for x in stream:
        if x in counters:
            counters[x] += 1
        elif len(counters) < k - 1:
            counters[x] = 1
        else:
            for key in list(counters):                  # decrement all; drop zeros
                counters[key] -= 1
                if counters[key] == 0:
                    del counters[key]
    return counters


def ref_parallel_top_k(items, k, num_procs):
    chunks = [items[i::num_procs] for i in range(num_procs)]
    total = {}
    with ProcessPoolExecutor(max_workers=num_procs) as ex:
        for d in ex.map(heavyhitters_workers.count_chunk, chunks):
            for key, v in d.items():
                total[key] = total.get(key, 0) + v
    return ref_top_k(total, k)


assert ref_count(["a", "a", "b"]) == {"a": 2, "b": 1}
assert ref_top_k({"a": 1, "b": 2, "c": 2}, 2) == [("b", 2), ("c", 2)]
_cc = RefConcurrentCounter(); _cc.add("z", 3); assert _cc.total("z") == 3 and _cc.top_k(1) == [("z", 3)]
_hh = ref_heavy_hitters(["a"] * 10 + ["b", "c"], 3); assert "a" in _hh and len(_hh) <= 2
assert ref_parallel_top_k(["a", "a", "a", "b", "b"], 1, 4) == [("a", 3)]
print("reference solutions OK")